# 🌀 Denoise (DeepInterpolation) → segment (suite2p) → compare — T4CT

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leonor-sinogas/T4CT-team-8/blob/main/notebooks/deepinterp_denoise.ipynb)

Whole comparison in one place:
1. **Denoise** a 2-photon movie with DeepInterpolation (Lecoq et al., *Nat. Methods* 2021).
2. **Segment** the raw movie and the denoised movie with suite2p.
3. **Compare** ROI count and trace SNR — the *intermediate* milestone.

`Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. Runs on a
bundled sample movie by default; point it at your TIFF in the *data* cell.

> **First run installs suite2p and restarts the runtime once** — just **Run all
> again** when it returns (it skips the install the second time).

In [ ]:
# 1) GPU check 
!nvidia-smi -L || echo 'No GPU — Runtime > Change runtime type > T4 GPU'

In [ ]:
# 2) Install DeepInterpolation + suite2p. TF/tifffile/h5py are already on Colab.
#    deepinterpolation: install from GitHub — the PyPI 0.2.0 release hard-imports
#    s3fs (which won't install cleanly on Colab); the GitHub version doesn't need it.
#    --no-deps reuses Colab's TensorFlow and avoids a tensorflow<->numba<->numpy
#    clash with suite2p; we add nibabel (its only dep not on Colab). suite2p forces
#    a ONE-TIME restart (numpy/numba); this cell restarts, then skips ahead next run.
import importlib.util, os, subprocess, sys

def _pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', *args], check=True)

need_di = importlib.util.find_spec('deepinterpolation') is None
need_s2p = importlib.util.find_spec('suite2p') is None
if need_di or need_s2p:
    if need_di:
        _pip('-q', '--no-deps', 'git+https://github.com/AllenInstitute/deepinterpolation.git')
        _pip('-q', 'nibabel')                          # its only dep not already on Colab
    if need_s2p:
        _pip('-q', 'suite2p')
    print('✅ Installed — restarting runtime; Run all again when it returns.')
    os.kill(os.getpid(), 9)
else:
    print('deps ready: deepinterpolation + suite2p')

In [ ]:
# 3) Pretrained 2-photon model (Allen 'Ai93': GCaMP6f, 512x512, 30 Hz)
import os
if not os.path.exists('ai93_model'):
    !wget -q -O ai93.zip 'https://www.dropbox.com/sh/vwxf1uq2j60uj9o/AAC9BQI1bdfmAL3OFO0lmVb1a?dl=1'
    !unzip -o -q ai93.zip -d ai93_model && rm ai93.zip
MODEL = 'ai93_model/2019_09_11_23_32_unet_single_1024_mean_absolute_error_Ai93-0450.h5'
print('model:', MODEL, '| exists:', os.path.exists(MODEL))

In [ ]:
# 4) Choose the movie to denoise
import tifffile, numpy as np
FPS = 30.0

# --- Option B (default): bundled sample 2p movie, so this runs without your data ---
if not os.path.exists('deepinterpolation'):
    !git clone -q --depth 1 https://github.com/AllenInstitute/deepinterpolation.git
TIF = 'deepinterpolation/sample_data/ophys_tiny_761605196.tif'

# --- Option A (your data): mount Drive and point at your recording ---
# from google.colab import drive; drive.mount('/content/drive')
# TIF = '/content/drive/MyDrive/T4CT/your_recording.tif'

print('movie:', TIF, '|', tifffile.TiffFile(TIF).series[0].shape)

In [ ]:
# 5) Denoise (TIFF -> denoised HDF5). end_frame=-1 does the whole movie.
from deepinterpolation.generator_collection import SingleTifGenerator
from deepinterpolation.inference_collection import core_inference

generator_param = dict(pre_post_frame=30, pre_post_omission=0, steps_per_epoch=-1,
                       train_path=TIF, batch_size=5,
                       start_frame=0, end_frame=-1,   # -1 = full movie
                       randomize=0)
inference_param = dict(model_path=MODEL, output_file='denoised.h5')
core_inference(inference_param, SingleTifGenerator(generator_param)).run()
print('done -> denoised.h5')

In [ ]:
# 6) Load raw + denoised, eyeball them, save a viewable denoised.tif
import h5py, matplotlib.pyplot as plt
raw = tifffile.imread(TIF).astype('float32')
with h5py.File('denoised.h5', 'r') as f:
    den = np.squeeze(f['data'][()]).astype('float32')
OFF = 30                          # denoised[i] <-> raw[OFF + i]
raw_matched = raw[OFF:OFF + len(den)]
i = len(den) // 2
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for a, img, t in zip(ax, [raw_matched[i], den[i], raw_matched[i] - den[i]],
                     ['noisy', 'DeepInterpolation', 'removed (noise)']):
    a.imshow(img, cmap='gray'); a.set_title(t); a.axis('off')
plt.show()
tifffile.imwrite('denoised.tif', den)
print('raw', raw.shape, '-> denoised', den.shape)
# Save to Drive (uncomment if mounted):
# import shutil; shutil.copy('denoised.tif', '/content/drive/MyDrive/T4CT/denoised.tif')

## Does denoising help segmentation? suite2p on raw vs denoised
We run suite2p on the **same frames** of the raw movie and of the denoised movie, then compare the number of detected cells and the trace SNR.

> The bundled sample movie is short (~few hundred frames; denoising drops 60), so suite2p may find few/no cells there — the comparison is meaningful on your **full recording**. For real numbers, use Option A with `end_frame=-1`.

In [ ]:
# 7) Reuse the project's tested suite2p wrapper
import sys
if not os.path.exists('T4CT-team-8'):
    !git clone -q https://github.com/leonor-sinogas/T4CT-team-8.git
sys.path.insert(0, 'T4CT-team-8/src')
from t4ct import segment, traces

# Pass arrays so the wrapper handles float->uint16 for suite2p.
out_raw = segment.run_suite2p(raw_matched, save_path='/content/s2p_raw', fs=FPS, tau=1.0)
out_den = segment.run_suite2p(den,        save_path='/content/s2p_den', fs=FPS, tau=1.0)
print('suite2p done for raw + denoised')

In [ ]:
# 8) Compare: # cells and trace SNR
def metrics(out):
    cells = out['iscell'][:, 0].astype(bool)
    dff = traces.dff(traces.neuropil_correct(out['F'][cells], out['Fneu'][cells]))
    noise = np.std(np.diff(dff, axis=1), axis=1) / np.sqrt(2)          # frame-to-frame noise
    peak = np.percentile(dff, 99, axis=1)                              # transient amplitude
    snr = peak / (noise + 1e-9)
    return int(cells.sum()), snr

n_raw, snr_raw = metrics(out_raw)
n_den, snr_den = metrics(out_den)
print(f'# cells      raw {n_raw:4d}   denoised {n_den:4d}')
if len(snr_raw) and len(snr_den):
    print(f'median SNR   raw {np.median(snr_raw):.2f}   denoised {np.median(snr_den):.2f}')

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(['raw', 'denoised'], [n_raw, n_den], color=['gray', 'C0'])
ax[0].set_title('# cells detected')
if len(snr_raw) and len(snr_den):
    ax[1].hist(snr_raw, bins=30, alpha=0.5, label='raw')
    ax[1].hist(snr_den, bins=30, alpha=0.5, label='denoised')
    ax[1].set_xlabel('trace SNR'); ax[1].set_title('trace SNR'); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# 9) ROI footprints side by side
from t4ct import viz
c_raw = out_raw['iscell'][:, 0].astype(bool)
c_den = out_den['iscell'][:, 0].astype(bool)
viz.plot_footprints(out_raw['footprints'][c_raw], out_raw['ops']['meanImg'], title='raw')
viz.plot_footprints(out_den['footprints'][c_den], out_den['ops']['meanImg'], title='denoised')

## Notes
- **Your data:** cell 4 Option A (mount Drive, set `TIF`); keep `end_frame=-1` in cell 5 for the full movie. Save `denoised.tif` to Drive (cell 6).
- **Reading the result:** more cells and/or higher SNR on the denoised movie = denoising helped. That delta is the intermediate milestone.
- **Pretrained vs your own:** Ai93 was trained on visual-cortex GCaMP6f; if results look off on your motor-cortex data, train your own (self-supervised) — see `docs/deepinterpolation.md`.
- **If install conflicts** between deepinterpolation and suite2p ever bite, do denoising here and the suite2p comparison in `T4CT_demo.ipynb` instead (load `denoised.tif` from Drive). See `docs/deepinterpolation.md`.